### Setup and imports

In [ ]:
import pandas as pd
import numpy as np
import rdkit
import ast
import matplotlib.pyplot as plt
# Fixed-grid multi-label logistic regression baseline.
from scipy.interpolate import interp1d
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import ast

#add future imports here

In [ ]:
labelled_df = pd.read_csv(r"data/IR spectroscopy data/SMILES_IR_Spectroscopy.csv", low_memory=False)

In [ ]:
def plot_graph(row):
    x_coords = row["x_coords"]
    y_coords = row["y_coords"]

    if isinstance(x_coords, str):
        x_coords = ast.literal_eval(x_coords)
    if isinstance(y_coords, str):
        y_coords = ast.literal_eval(y_coords)

    plt.figure(figsize=(10, 4))
    plt.plot(x_coords, y_coords, linewidth=1)
    plt.xlabel("1/cm")
    plt.ylabel("Absorbance")
    plt.title(row.get("smiles", "IR Spectrum"))
    plt.tight_layout()
    plt.show()
    return None

In [ ]:
#test it
plot_graph(labelled_df.iloc[0])
for i in range(15):
    print("================")
    plot_graph(labelled_df.iloc[i])

## Normalization of data

### Standardize spectra through interpolating coordinate data to fixed grid 
* Using indiviail based normalizaiton and not group global normalization because:
  * experimental conditions may not have been consistant
  * the model learns scale artifacts (instrument, concentration) which makes it worse at generalization

In [ ]:
#temp. perform normalization on small sample of data for computation
test_df = labelled_df.sample(500)

In [ ]:
#test it
for i in range(5):
    print(f'====regular{i}=====')
    plot_graph(labelled_df.iloc[i])
    print(f'====normalized{i}=====')
    plot_graph(normalized_y_df.iloc[i])
    print(f'===================')

## Labeling

### Canonicallize smiles 
* learn more at 
  * https://luis-vollmers.medium.com/tutorial-to-smiles-and-canonical-smiles-explained-with-examples-fbc8a46ca29f

In [ ]:
#Smiles canonicalization implementation
from rdkit import Chem

def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol)
    except:
        return None

normalized_y_df['canonical_smiles'] = normalized_y_df['smiles'].apply(canonicalize_smiles)
canonicalized_smiles_df = normalized_y_df
print(f"Successfully canonicalized: {canonicalized_smiles_df['canonical_smiles'].notna().sum()} our of {len(canonicalized_smiles_df)}")
canonicalized_smiles_df.head()

### Create Labels 
* using martini 3 mapping algorithm to label compounds from smiles
* refer to paper 

In [12]:
# Load the labelled data and add derivative channels for the CNN.
labelled_df = pd.read_csv("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/Labelled_SMILES_IR_Spectroscopy.csv")

## Baseline Modeling

* set an initial comparison point for assessing the performance of more complex models.

## Main Model

### Train a 1D CNN with Multi-Task Outputs
* hyper parameters and justification
  * using cross entropy loss function for multi classification
  * in channels: spectroscopy (x,y) pair coordinate values, first derivative of coordinate, second derivative of coordinate 
  * out channel: list of beads with probability attached. Note there could be multiple beeds and multiple beed of the same type ex: [N: 0.67, C5: 0.23, C5: 0.02, N4: 0.93, C5: 0.21, C5: 0.11, C5: 0.87, C5: 0.21]
  * kernel size sohuld very and we will see which is best with evaluations
  * validtion dataset will be used to help backpropagate parameters
  

In [14]:
from sklearn.preprocessing import MultiLabelBinarizer

labelled_df['bead_list'] = labelled_df['bead_summary'].apply(lambda x: [b.strip() for b in x.split(',')] if isinstance(x, str) else [])

mlb = MultiLabelBinarizer()
bead_labels = mlb.fit_transform(labelled_df['bead_list'])

print(f"Number of unique bead types: {len(mlb.classes_)}")
print(f"Bead types: {mlb.classes_}")
print(f"Label matrix shape: {bead_labels.shape}")

Number of unique bead types: 23
Bead types: ['C1' 'C2' 'C3' 'C4' 'C5' 'C6' 'N1' 'N2' 'N3' 'N4' 'N5' 'N6' 'P1' 'P2'
 'P3' 'P4' 'P5' 'P6' 'Q4' 'X1' 'X2' 'X3' 'X4']
Label matrix shape: (7910, 23)


In [19]:
def interpolate_spectrum(row,target_length=3600):
    x = ast.literal_eval(row['x_coords']) if isinstance(row['x_coords'], str) else row['x_coords']
    y = ast.literal_eval(row['y_coords']) if isinstance(row['y_coords'], str) else row['y_coords']
    f = interp1d(x, y, bounds_error=False, fill_value=0)
    x_new = np.linspace(400, 4000, target_length)
    return f(x_new)
    
spectra = np.array([interpolate_spectrum(row) for _, row in labelled_df.iterrows()])
print(f"Spectra shape: {spectra.shape}")

Spectra shape: (7910, 3600)


### Data Splitting
* 80/20 train/test split on labelled data.

In [20]:
X = torch.tensor(spectra, dtype=torch.float32).unsqueeze(1)  
y = torch.tensor(bead_labels, dtype=torch.float32)           # Shape: (7910, 23)

# Train/test split
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset)) #train sample size would be 80% of the dataset
test_size = len(dataset) - train_size #test sample size would be 20% of the dataset
train_set, test_set = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32)

print(f"Training samples: {train_size}")
print(f"Test samples: {test_size}")


Training samples: 6328
Test samples: 1582


In [21]:
class IRBeadClassifier(nn.Module):
    def __init__(self, input_length, num_bead_types):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )
        conv_output_size = 128 * (input_length // 8)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_output_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_bead_types),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.classifier(x)

model = IRBeadClassifier(input_length=3600, num_bead_types=23)
print(model)


IRBeadClassifier(
  (conv_layers): Sequential(
    (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=57600, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=23, bias=True)
    (5): Sigmoid()
  )
)


In [22]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")


Epoch 1/30 | Loss: 0.2508


KeyboardInterrupt: 

## Evaluation
* for results section

In [31]:
model = IRBeadClassifier(input_length=3600, num_bead_types=23)
state_dict = torch.load("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/model/ir_bead_classifier.pth")
model.load_state_dict(state_dict)
print(model)

IRBeadClassifier(
  (conv_layers): Sequential(
    (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=57600, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=23, bias=True)
    (5): Sigmoid()
  )
)


### Evaluate Functional Group Prediction (F1, Precision, Recall)

### Evaluate Retrieval Performance (Top-K Accuracy, MRR)

### Calibration and Confidence Analysis

## Deployment Design

### Functional Group Probability Outputs

### Top-K SMILES Candidate Retrieval

### Confidence and Uncertainty Display